In [ ]:
# Colab setup: upload the Excel dataset
!pip -q install openpyxl

from google.colab import files
uploaded = files.upload()
file_name = next(iter(uploaded))
print('Uploaded:', file_name)


# Thiranex Internship — Task 4
## Real-World Retail Sales Prediction & Business Analytics

**Domain:** Retail  
**Objective:** Perform an end-to-end analysis of real-world retail transactions and build a machine-learning model to predict daily revenue.

## 1. Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

## 2. Load Dataset

In [ ]:
import pandas as pd

df = pd.read_excel(file_name)
print('Original shape:', df.shape)
df.head()


## 3. Data Cleaning

Cleaning rules:
- Convert `InvoiceDate` to datetime.
- Remove duplicate rows.
- Remove rows missing fields required for transaction analysis.
- Keep positive quantities and positive unit prices.
- Calculate `Revenue = Quantity × UnitPrice`.

In [ ]:
df["InvoiceDate"] = pd.to_datetime(df["InvoiceDate"], errors="coerce")
print("Missing values:")
print(df.isnull().sum())
print("Duplicate rows:", df.duplicated().sum())

df = df.drop_duplicates()
df = df.dropna(subset=["InvoiceNo","StockCode","InvoiceDate","Quantity","UnitPrice","Country"])
df = df[(df["Quantity"] > 0) & (df["UnitPrice"] > 0)].copy()
df["Revenue"] = df["Quantity"] * df["UnitPrice"]
print("Cleaned shape:", df.shape)

## 4. Feature Engineering

In [ ]:
df["Year"] = df["InvoiceDate"].dt.year
df["Month"] = df["InvoiceDate"].dt.month
df["Day"] = df["InvoiceDate"].dt.day
df["DayOfWeek"] = df["InvoiceDate"].dt.dayofweek
df["Hour"] = df["InvoiceDate"].dt.hour
df.head()

## 5. Exploratory Data Analysis

In [ ]:
monthly_revenue = df.groupby(df["InvoiceDate"].dt.to_period("M"))["Revenue"].sum()
plt.figure(figsize=(11,5))
plt.plot(monthly_revenue.index.astype(str), monthly_revenue.values)
plt.xticks(rotation=60)
plt.title("Monthly Revenue Trend")
plt.xlabel("Month"); plt.ylabel("Revenue (£)")
plt.tight_layout(); plt.show()

In [ ]:
top_products = df.groupby("Description")["Revenue"].sum().sort_values(ascending=False).head(10)
plt.figure(figsize=(10,5))
top_products.sort_values().plot(kind="barh")
plt.title("Top 10 Products by Revenue"); plt.xlabel("Revenue (£)")
plt.tight_layout(); plt.show()

In [ ]:
country_revenue = df.groupby("Country")["Revenue"].sum().sort_values(ascending=False).head(10)
plt.figure(figsize=(10,5))
country_revenue.sort_values().plot(kind="barh")
plt.title("Top 10 Countries by Revenue"); plt.xlabel("Revenue (£)")
plt.tight_layout(); plt.show()

## 6. Business Metrics

In [ ]:
metrics = pd.Series({
    "Total Revenue": df["Revenue"].sum(),
    "Total Units Sold": df["Quantity"].sum(),
    "Unique Invoices": df["InvoiceNo"].nunique(),
    "Unique Products": df["StockCode"].nunique(),
    "Countries": df["Country"].nunique()
})
metrics

## 7. Build Daily Sales Dataset

Daily revenue is the prediction target. Lag and rolling features use previous observations only.

In [ ]:
daily = (df.set_index("InvoiceDate").resample("D").agg(
    Revenue=("Revenue","sum"),
    Units=("Quantity","sum"),
    Transactions=("InvoiceNo","nunique"),
    Products=("StockCode","nunique"),
    Customers=("CustomerID","nunique")
).reset_index())

for c in ["Revenue","Units","Transactions","Products","Customers"]:
    daily[c] = daily[c].fillna(0)

daily["Day"] = daily["InvoiceDate"].dt.day
daily["Month"] = daily["InvoiceDate"].dt.month
daily["Year"] = daily["InvoiceDate"].dt.year
daily["DayOfWeek"] = daily["InvoiceDate"].dt.dayofweek
daily["IsWeekend"] = (daily["DayOfWeek"] >= 5).astype(int)
daily["Lag_1"] = daily["Revenue"].shift(1)
daily["Lag_7"] = daily["Revenue"].shift(7)
daily["Rolling_7"] = daily["Revenue"].shift(1).rolling(7).mean()
daily = daily.dropna().copy()
daily.head()

## 8. Chronological Train/Test Split

In [ ]:
feature_cols = ["Day","Month","Year","DayOfWeek","IsWeekend","Lag_1","Lag_7","Rolling_7","Units","Transactions","Products","Customers"]
split = int(len(daily) * 0.8)
train = daily.iloc[:split]
test = daily.iloc[split:]
X_train, y_train = train[feature_cols], train["Revenue"]
X_test, y_test = test[feature_cols], test["Revenue"]
print("Training rows:", len(train))
print("Testing rows:", len(test))

## 9. Random Forest Regression

In [ ]:
model = RandomForestRegressor(
    n_estimators=250, random_state=42, n_jobs=-1,
    max_depth=12, min_samples_leaf=2
)
model.fit(X_train, y_train)
predictions = model.predict(X_test)

## 10. Model Evaluation

In [ ]:
mae = mean_absolute_error(y_test, predictions)
rmse = np.sqrt(mean_squared_error(y_test, predictions))
r2 = r2_score(y_test, predictions)
print(f"MAE: £{mae:,.2f}")
print(f"RMSE: £{rmse:,.2f}")
print(f"R² Score: {r2:.4f}")

## 11. Actual vs Predicted Revenue

In [ ]:
plt.figure(figsize=(11,5))
plt.plot(test["InvoiceDate"], y_test.values, label="Actual")
plt.plot(test["InvoiceDate"], predictions, label="Predicted")
plt.title("Actual vs Predicted Daily Revenue")
plt.xlabel("Date"); plt.ylabel("Revenue (£)")
plt.legend(); plt.tight_layout(); plt.show()

## 12. Feature Importance

In [ ]:
importance = pd.Series(model.feature_importances_, index=feature_cols).sort_values(ascending=False)
plt.figure(figsize=(9,5))
importance.sort_values().plot(kind="barh")
plt.title("Random Forest Feature Importance")
plt.xlabel("Importance"); plt.tight_layout(); plt.show()
importance

## 13. Key Findings

- The raw retail dataset was transformed into a cleaned analytical dataset.
- Revenue patterns were analysed across time, products, and countries.
- Daily historical and calendar features were engineered for prediction.
- A Random Forest regression model was trained using a chronological holdout set.
- MAE, RMSE, and R² were used to evaluate prediction performance.
- Actual-vs-predicted and feature-importance plots provide model interpretation.

The numerical metrics should be taken from the executed notebook output.

## 14. Conclusion

This project demonstrates an end-to-end real-world data science workflow: data cleaning, exploratory analysis, feature engineering, business analysis, machine-learning regression, and model evaluation.

Possible future improvements include advanced time-series forecasting, customer segmentation, product-level demand forecasting, hyperparameter tuning, and interactive dashboards.

## Dataset Source

UCI Machine Learning Repository — Online Retail  
https://archive.ics.uci.edu/dataset/352/online+retail